In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from joblib import dump
import warnings
warnings.filterwarnings("ignore") #remove convergence warnings

In [4]:
data = pd.read_csv("forestfires.csv")

features = ["X", "Y", "month", "day", "temp", "RH", "wind", "rain"]
target = "area"

#drop rows with missing values
data = data.dropna(subset=features + [target]).copy()

#one-hot encode month and day
data = pd.get_dummies(data, columns=["month", "day"], drop_first=False)

X = data.drop(columns=[target]).values #features
y = data[target].values #target (area)

#reduce skewness of target as seen in paper
y_trans = np.log(y + 1.0)

In [5]:
#SVM parameter setup from paper

N = len(X)
sigma_hat = np.std(y_trans)
epsilon_heur = 3.0 * sigma_hat * np.sqrt(np.log(N) / N)
C_heur = 3.0

param_grid = {
    "gamma": [2.0**-9, 2.0**-7, 2.0**-5, 2.0**-3, 2.0**-1]
}

In [6]:
#nested grid search with 30 runs of 10-fold cross-validation

num_runs = 30
num_folds = 10

mae_all_runs = []
rmse_all_runs = []
r2_all_runs = []

for run in range(num_runs):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=run)

    mae_folds = []
    rmse_folds = []
    r2_folds = []

    for train_idx, test_idx in kf.split(X):
        #split the dataset into train and test for this fold
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train_trans, y_test_trans = y_trans[train_idx], y_trans[test_idx]
        y_test = y[test_idx]  # We'll use these to measure actual error (in original units)

        #standardize the features in both train and test sets
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        #initiaize the support vector regression model with fixed C and epsilon
        base_svr = SVR(kernel="rbf", C=C_heur, epsilon=epsilon_heur)

        #perform a grid search on the training set to find the best gamma
        grid_search = GridSearchCV(
            estimator=base_svr,
            param_grid=param_grid,
            scoring="neg_mean_absolute_error",
            cv=5,
            refit=True
        )
        grid_search.fit(X_train, y_train_trans)

        #get the best performing model
        best_svr = grid_search.best_estimator_

        #evaluate the model on the test set
        pred_test_log = best_svr.predict(X_test)
        #reverse the log transform
        pred_test = np.exp(pred_test_log) - 1.0

        #calculate errors
        mae_fold = np.mean(np.abs(pred_test - y_test))
        rmse_fold = np.sqrt(np.mean((pred_test - y_test) ** 2))
        r2_fold = r2_score(y_test, pred_test)

        mae_folds.append(mae_fold)
        rmse_folds.append(rmse_fold)
        r2_folds.append(r2_fold)

    #average errors across current fold
    run_mae = np.mean(mae_folds)
    run_rmse = np.mean(rmse_folds)
    run_r2 = np.mean(r2_folds)

    mae_all_runs.append(run_mae)
    rmse_all_runs.append(run_rmse)
    r2_all_runs.append(run_r2)

#average errors across all runs
mae_avg = np.mean(mae_all_runs)
rmse_avg = np.mean(rmse_all_runs)
r2_avg = np.mean(r2_all_runs)

#standard deviation across all runs
mae_std = np.std(mae_all_runs, ddof=1)
rmse_std = np.std(rmse_all_runs, ddof=1)
r2_std = np.std(r2_all_runs, ddof=1)

#1.96 for 95% confidence interval
num_runs_float = float(num_runs)
ci_factor = 1.96 / np.sqrt(num_runs_float)

print(f"STM subset (30 runs x 10-fold CV)")
print(f"Mean MAE (MAD):  {mae_avg:.3f} ± {ci_factor * mae_std:.3f}")
print(f"Mean RMSE:       {rmse_avg:.3f} ± {ci_factor * rmse_std:.3f}")
print(f"Mean R^2:        {r2_avg:.3f} ± {ci_factor * r2_std:.3f}")

STM subset (30 runs x 10-fold CV)
Mean MAE (MAD):  12.887 ± 0.035
Mean RMSE:       46.296 ± 0.789
Mean R^2:        -0.124 ± 0.008


In [7]:
#train a final model on the entire dataset

#standardize the features
X_df = data.drop(columns=[target])
scaler_final = StandardScaler()
X_scaled_full = scaler_final.fit_transform(X_df)

#initialize the support vector regression model with fixed C and epsilon
base_svr_final = SVR(kernel="rbf", C=C_heur, epsilon=epsilon_heur)

#perform a grid search on the entire dataset to find the best gamma
grid_search_final = GridSearchCV(
    estimator=base_svr_final,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    refit=True
)
grid_search_final.fit(X_scaled_full, y_trans)

#get the best performing model
best_svr_final = grid_search_final.best_estimator_

#combine scaler and SVR model into a pipeline for easier deployment
final_pipeline = Pipeline([
    ('scaler', scaler_final),
    ('svr', best_svr_final)
])

#evaluate the final model on the full dataset
y_pred_final = final_pipeline.predict(X)

y_pred_orig = np.exp(y_pred_final) - 1
y_actual_orig = np.exp(y_trans) - 1

mae_final_orig = mean_absolute_error(y_actual_orig, y_pred_orig)
rmse_final_orig = np.sqrt(mean_squared_error(y_actual_orig, y_pred_orig))
r2_final_orig = r2_score(y_actual_orig, y_pred_orig)

print(f"MAE: {mae_final_orig:.3f} hectares")
print(f"RMSE: {rmse_final_orig:.3f} hectares")
print(f"R^2: {r2_final_orig:.3f}")

#save the final pipeline
dump(final_pipeline, "final_pipeline.joblib")

print("Final scaler and SVR model saved as 'final_pipeline.joblib'.")

MAE: 12.647 hectares
RMSE: 64.684 hectares
R^2: -0.035
Final scaler and SVR model saved as 'final_pipeline.joblib'.
